In [11]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
print(torch.cuda.is_available())

False


In [ ]:
# Step 2: Load ONE image and inspect it
sample_path = "dataset/train/no/2.npy"  # one .npy file

image = np.load(sample_path)

print("Shape:", image.shape)   # e.g. (150, 150) or (150, 150, 1)
print("Min value:", image.min())
print("Max value:", image.max())
print("Data type:", image.dtype)

In [ ]:
# Step 3: See what the image actually looks like
plt.imshow(image.squeeze(), cmap='gray')  
# .squeeze() removes dimensions of size 1, e.g. (150,150,1) → (150,150)
plt.title("Sample Gravitational Lens Image")
plt.colorbar()
plt.show()

In [ ]:
import os

data_dir = "dataset/train"  # change this

all_paths = []
all_labels = []

# os.listdir gives you the folder names — which are your class names
classes = sorted([
    cls for cls in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, cls))
])
# sorted() so order is consistent every run

# Build a dictionary: class name → number
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

print("Classes found:", classes)
print("Class to index:", class_to_idx)

# Now walk through each class folder
for cls in classes:
    cls_folder = os.path.join(data_dir, cls)
    for filename in os.listdir(cls_folder):
        filepath = os.path.join(cls_folder, filename)
        all_paths.append(filepath)
        all_labels.append(class_to_idx[cls])

print(f"\nTotal images found: {len(all_paths)}")
print("First 3 paths:", all_paths[:3])
print("First 3 labels:", all_labels[:3])

In [ ]:
from collections import Counter

label_counts = Counter(all_labels)

print(label_counts)

In [ ]:
val_dir = "dataset/val"

val_paths = []
val_labels = []

for cls in classes:
    cls_folder = os.path.join(val_dir, cls)
    for filename in os.listdir(cls_folder):
        filepath = os.path.join(cls_folder, filename)
        val_paths.append(filepath)
        val_labels.append(class_to_idx[cls])

print(f"Total val images: {len(val_paths)}")

# Check class balance
from collections import Counter
print("Val class distribution:", Counter(val_labels))

In [ ]:
from sklearn.model_selection import train_test_split


val_paths, test_paths, val_labels, test_labels = train_test_split(
    val_paths,
    val_labels,
    test_size=0.5,        # 50% goes to test
    random_state=42,      # fixed seed so split is same every run
    stratify=val_labels   # keep class balance in both halves
)

print(f"Val size: {len(val_paths)}")
print(f"Test size: {len(test_paths)}")
print("Val distribution:", Counter(val_labels))
print("Test distribution:", Counter(test_labels))

1. Data Loading & EDA (don't skip this)

Visualize samples from all 3 classes — subhalo and vortex substructure have distinct visual signatures, mention what you observe
Check class balance — if imbalanced, note it and address it
The data is already min-max normalized, but check if you want to re-normalize to ImageNet stats if using a pretrained backbone

2. Augmentation Strategy
This is where you show judgment. For lensing images specifically:

Random horizontal/vertical flips ✓ (lensing is rotationally symmetric)
Random rotations (90°, 180°, 270° or continuous) ✓ — physically motivated
Avoid augmentations that break physics — like aggressive color jitter or cutout near the lens center
Mention why each augmentation is physically valid — this is the "discuss your strategy" part

3. Model Choice
Go with EfficientNet-B0 pretrained on ImageNet. Reasons to state:

Strong accuracy/parameter tradeoff
Transfer learning from ImageNet still helps even for non-natural images at these scales
Alternatively ResNet18 is fine too, slightly easier to explain

Modify the final classifier head for 3-class output.
4. Training Setup

90/10 stratified train-test split (they specify this)
CrossEntropyLoss
AdamW optimizer with weight decay
Cosine LR scheduler or ReduceLROnPlateau
Early stopping on validation loss
Mixed precision (fp16) if you want to show modern training hygiene

5. Evaluation — this is what they actually grade

ROC curve: must be one-vs-rest for each of the 3 classes, all plotted together
AUC score: report per-class AUC and macro-average AUC
Also include confusion matrix and per-class accuracy as supplementary — shows thoroughness even though not required

6. The "discuss your strategy" section
This is explicitly asked for and most people write two lines. Write a proper paragraph covering:

Why you chose this architecture
Why transfer learning makes sense here
What your augmentation choices are physically motivated by
What the key challenge is (subtle substructure differences between subhalo and vortex)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=180),  # continuous, not just 90/180/270
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], 
                        std=[0.5, 0.5, 0.5])
])

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
